# RMSnorm

$$
y = \frac{x}{\sqrt{\text{mean}(x^2)+\epsilon}}\odot w
$$

与 LayerNorm 相比，RMSNorm 省去了均值中心化与偏置项。Forward 只需保存每行的
rstd，backward 复用它，避免重新计算统计量。`dx` 中均值对应的梯度项（`c2`）消失，
但仍含一个行内的 dot 项；`dw` 依然需要汇总所有行的贡献，因此 partial buffer 与
lock 的归约设计可以原样复用。

In [1]:
# 首先用torch手动实现一个RMSnorm kernel

import torch

def rmsnorm_reference(
  x: torch.Tensor,
  weight: torch.Tensor,
  eps: float=1e-6,
) -> torch.Tensor:
  x_fp32 = x.float()
  var = x_fp32.pow(2).mean(
    dim=-1,
    keepdim=True,
  )
  
  x_norm = x_fp32 * torch.rsqrt(var+eps)
  y = x_norm * weight.float()
  return y.to(x.dtype)


def main():
  seed = 990808
  torch.manual_seed(seed)

  M, N = 1024, 64
  eps = 1e-6

  x = torch.randn((M,N), device='cuda',dtype=torch.float16)
  w = torch.randn((N,), device='cuda',dtype=torch.float16)

  y_ref = rmsnorm_reference(
    x, w, eps
  )

  # control
  torch_rms = torch.nn.RMSNorm(
        normalized_shape=N, # RMSNorm 对最后一个维度进行归一化
        eps=eps,
        elementwise_affine=True,
        device=x.device,
        dtype=x.dtype,)
  
  with torch.no_grad():
    torch_rms.weight.copy_(w)
  
  y_torch = torch_rms(x)

  # ---------------------------
  # Correctness check
  # ---------------------------
  print("max abs error:",
        (y_ref - y_torch).abs().max().item())

  print("mean abs error:",
        (y_ref - y_torch).abs().mean().item())

  print(
      "allclose:",
      torch.allclose(
          y_ref,
          y_torch,
          atol=1e-3,
          rtol=1e-3,
      ),
  )

if __name__ == "__main__":
    main()

max abs error: 0.0
mean abs error: 0.0
allclose: True


In [6]:
# 接下来我们手写一个 triton 的 RMSnorm kernel；

import torch
import triton
import triton.language as tl 

@triton.jit
def rmsnorm_kernel(
  x_ptr, 
  w_ptr,
  y_ptr,
  n_cols: tl.constexpr,
  eps: tl.constexpr,
  BLOCK_SIZE: tl.constexpr,
):
  row = tl.program_id(0)
  offs = tl.arange(0, BLOCK_SIZE)
  mask = offs < n_cols

  x = tl.load (
    x_ptr + row * n_cols + offs, mask=mask, other=0.0
  )

  square = x*x
  var = tl.sum(square)/n_cols
  rstd = tl.rsqrt(var + eps)

  w = tl.load(w_ptr+offs, mask=mask, other=0.0)

  y = x * rstd * w

  tl.store(y_ptr+row*n_cols+offs, y, mask=mask)


def rmsnorm_triton(
    x,
    weight,
    eps=1e-6,
):
  row, col = x.shape
  col_ = weight.shape[0]
  assert col == col_
  assert x.is_contiguous()
  assert weight.is_contiguous()

  BLOCK_SIZE = triton.next_power_of_2(col)
  y = torch.empty_like(x)

  rmsnorm_kernel[(row,)](
    x, weight, y, col, eps, BLOCK_SIZE,
    num_warps=4,
  )

  return y


def main():
  seed = 990808
  torch.manual_seed(seed)

  M, N = 1024, 64
  eps = 1e-6

  x = torch.randn((M,N), device='cuda',dtype=torch.float16)
  w = torch.randn((N,), device='cuda',dtype=torch.float16)

  y_ref = rmsnorm_triton(
    x, w, eps
  )

  # control
  torch_rms = torch.nn.RMSNorm(
        normalized_shape=N, # RMSNorm 对最后一个维度进行归一化
        eps=eps,
        elementwise_affine=True,
        device=x.device,
        dtype=x.dtype,)
  
  with torch.no_grad():
    torch_rms.weight.copy_(w)
  
  y_torch = torch_rms(x)

  # ---------------------------
  # Correctness check
  # ---------------------------
  print("max abs error:",
        (y_ref - y_torch).abs().max().item())

  print("mean abs error:",
        (y_ref - y_torch).abs().mean().item())

  print(
      "allclose:",
      torch.allclose(
          y_ref,
          y_torch,
          atol=1e-3,
          rtol=1e-3,
      ),
  )

if __name__ == "__main__":
    main()

max abs error: 0.0078125
mean abs error: 0.00010102987289428711
allclose: True


In [15]:
def test_rmsnorm():
    torch.manual_seed(990808)

    shapes = [
        (1, 128),
        (32, 768),
        (128, 1024),
        (512, 4096),
        (1024, 8192),
    ]

    dtypes = [
        torch.float16,
        torch.bfloat16,
        torch.float32,
    ]

    for M, N in shapes:
        for dtype in dtypes:

            x = torch.randn(
                M,
                N,
                device="cuda",
                dtype=dtype,
            )

            weight = torch.randn(
                N,
                device="cuda",
                dtype=dtype,
            )

            ref = rmsnorm_reference(
                x,
                weight,
            )

            out = rmsnorm_triton(
                x,
                weight,
            )

            if dtype == torch.float32:
                atol = rtol = 1e-5
            else:
                atol = rtol = 1e-2

            # torch.testing.assert_close(
            #     out,
            #     ref,
            #     atol=atol,
            #     rtol=rtol,
            #     msg=f'({M}, {N}) at {dtype}'
            # )
          
            diff = (out.float() - ref.float()).abs()

            tolerance = atol + rtol * ref.float().abs()
            failed = diff > tolerance

            print(f"shape: ({M}, {N}), dtype: {dtype}")
            print(f"max abs error:  {diff.max().item()}")
            print(f"mean abs error: {diff.mean().item()}")
            print(f"failed count:   {failed.sum().item()}")
            print(f"total count:    {failed.numel()}")
            print(
                "failed ratio:   "
                f"{failed.float().mean().item():.8%}"
            )
            print(f"out has NaN:    {torch.isnan(out).any().item()}")
            print(f"ref has NaN:    {torch.isnan(ref).any().item()}")

            flat_index = diff.argmax()

            print("worst out:", out.flatten()[flat_index].item())
            print("worst ref:", ref.flatten()[flat_index].item())
            print("worst diff:", diff.flatten()[flat_index].item())
            print(
                "worst tolerance:",
                tolerance.flatten()[flat_index].item(),
            )

test_rmsnorm()

shape: (1, 128), dtype: torch.float16
max abs error:  0.0009765625
mean abs error: 4.443526268005371e-05
failed count:   0
total count:    128
failed ratio:   0.00000000%
out has NaN:    False
ref has NaN:    False
worst out: -1.634765625
worst ref: -1.6357421875
worst diff: 0.0009765625
worst tolerance: 0.026357421651482582
shape: (1, 128), dtype: torch.bfloat16
max abs error:  0.0078125
mean abs error: 0.00013065338134765625
failed count:   0
total count:    128
failed ratio:   0.00000000%
out has NaN:    False
ref has NaN:    False
worst out: 1.4453125
worst ref: 1.453125
worst diff: 0.0078125
worst tolerance: 0.024531248956918716
shape: (1, 128), dtype: torch.float32
max abs error:  0.0
mean abs error: 0.0
failed count:   0
total count:    128
failed ratio:   0.00000000%
out has NaN:    False
ref has NaN:    False
worst out: -0.2541956603527069
worst ref: -0.2541956603527069
worst diff: 0.0
worst tolerance: 1.2541956493805628e-05
shape: (32, 768), dtype: torch.float16
max abs error

In [17]:
# fix the M, scan different N

import time
import pandas as pd

def _bench(fn, n_repeat=50):
    # 预热，避免首次启动/cuDNN 等开销计入计时
    for _ in range(10):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_repeat):
        fn()
    torch.cuda.synchronize()
    # 单位换算为微秒 (μs)
    return (time.perf_counter() - t0) / n_repeat * 1e6


def main():
  seed = 990808
  torch.manual_seed(seed)

  M = 1024
  Ns = [128, 256, 512, 768, 1024,
        2048, 4096, 8192, 16384]
  eps = 1e-6

  dtypes = [torch.float16, torch.bfloat16, torch.float32]

  rows = []
  for N in Ns:
    for dtype in dtypes:
      # 固定 shape 后在计时循环外分配输入，避免 malloc 污染计时
      x = torch.randn((M, N), device='cuda', dtype=dtype).contiguous()
      weight = torch.randn((N,), device='cuda', dtype=dtype).contiguous()

      t_triton = _bench(lambda: rmsnorm_triton(x, weight))
      t_torch = _bench(lambda: torch.nn.functional.rms_norm(x, (N,), weight, eps))

      y_triton = rmsnorm_triton(x, weight)
      y_torch = torch.nn.functional.rms_norm(x, (N,), weight, eps)
      max_error = (y_triton.float() - y_torch.float()).abs().max().item()

      # 有效带宽：读写 x/y 各一次 + 读 weight 一次（近似）
      bytes_per = x.numel() * x.element_size() * 2 + N * x.element_size()
      eff_gbps = bytes_per / (t_triton * 1e-6) / 1e9

      rows.append({
        "M": M, "N": N, "dtype": str(dtype).split('.')[-1],
        "Triton (μs)": round(t_triton, 3),
        "Torch (μs)": round(t_torch, 3),
        "Speedup": round(t_torch / t_triton, 2),
        "Effective GB/s": round(eff_gbps, 2),
        "max error": max_error,
      })

  df = pd.DataFrame(rows)
  #print(df.to_markdown(index=False))
  return df

main()

,M,N,dtype,Triton (μs),Torch (μs),Speedup,Effective GB/s,max error
0,1024,128,float16,55.409,24.198,0.44,9.47,7.812500e-03
1,1024,128,bfloat16,46.377,22.051,0.48,11.31,6.250000e-02
2,1024,128,float32,73.415,25.670,0.35,14.29,1.907349e-06
3,1024,256,float16,49.803,16.415,0.33,21.06,3.906250e-03
4,1024,256,bfloat16,44.856,17.425,0.39,23.39,6.250000e-02
5,1024,256,float32,36.024,18.897,0.52,58.24,1.430511e-06
6,1024,512,float16,44.540,17.542,0.39,47.11,7.812500e-03
7,1024,512,bfloat16,35.240,17.981,0.51,59.54,6.250000e-02
8,1024,512,float32,36.585,25.248,0.69,114.70,1.907349e-06
9,1024,768,float16,34.130,21.344,0.63,92.21,7.812500e-03


## 一个重要问题：x 是否真的只从 HBM 读取一次？

在上述代码中：

```python
x = tl.load(...)
```

然后整个：square -> sum -> rstd -> multiply 流程都基于这个`x`。

理想情况下，`x` 在 program 生命周期内保留在 register/on-chip working set 中，因此无需第二次从 HBM 加载。
这也是 fused RMSNorm 很有价值的地方。
但是，这并不意味着 hidden size 可以无限增大。

## Hidden dimension 太大会发生什么？

目前我们的hidden dimension `N` = 64，是一个比较小的值；当 `N` = 4096时，一个program 保存 4096 个 FP32 values：
$$
4096\times4=16\,KB
$$
只是 x 就已经对应约 16 KB logical data。

但如果`N` = 16384，则对应为 64 KB。再加：
- reduction temporary；
- weight；
- intermediate；
- compiler-generated values；

register pressure 会明显增大。
官方 Triton LayerNorm 教程明确给 forward fused implementation 设置了 feature dimension 小于 64 KB per feature 的约束，超出时直接拒绝该 fused path。